In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_disease_dataset.csv")
df["date"] = pd.to_datetime(df["date"])
print(df.shape)
df.head()

(1040, 18)


,village_id,village_name,latitude,longitude,population,water_source_type,date,ph_level,turbidity_ntu,residual_chlorine_mg_l,bacterial_contamination,rainfall_mm,temperature_c,sanitation_index,historical_cases_last_4_weeks,disease_type,case_count,outbreak_risk_level
0,V001,Village_1,16.57454,74.31185,10470,pond,2024-01-07,6.84,1.95,0.456,0,13.1,32.4,0.34,0,diarrhea,8,High
1,V001,Village_1,16.57454,74.31185,10470,pond,2024-01-14,7.33,2.31,0.462,0,6.4,25.7,0.34,8,typhoid,18,High
2,V001,Village_1,16.57454,74.31185,10470,pond,2024-01-21,5.94,2.49,0.326,0,5.4,28.9,0.34,18,diarrhea,6,Medium
3,V001,Village_1,16.57454,74.31185,10470,pond,2024-01-28,6.58,1.78,0.382,0,8.6,28.0,0.34,6,cholera,2,Low
4,V001,Village_1,16.57454,74.31185,10470,pond,2024-02-04,6.99,2.84,0.274,0,7.0,37.4,0.34,2,hepatitis_a,6,Medium


In [5]:
df["water_quality_risk_score"] = (
    (df["turbidity_ntu"] / 10) +
    (1 - df["residual_chlorine_mg_l"]) +
    (df["bacterial_contamination"] * 1.0)
)

df[["turbidity_ntu", "residual_chlorine_mg_l", "bacterial_contamination", "water_quality_risk_score"]].head()

,turbidity_ntu,residual_chlorine_mg_l,bacterial_contamination,water_quality_risk_score
0,1.95,0.456,0,0.739
1,2.31,0.462,0,0.769
2,2.49,0.326,0,0.923
3,1.78,0.382,0,0.796
4,2.84,0.274,0,1.010


In [6]:
df["environmental_risk_score"] = (
    (df["rainfall_mm"] / 50) +
    ((df["temperature_c"] - 20) / 15)
)

df[["rainfall_mm", "temperature_c", "environmental_risk_score"]].head()

,rainfall_mm,temperature_c,environmental_risk_score
0,13.1,32.4,1.088667
1,6.4,25.7,0.508000
2,5.4,28.9,0.701333
3,8.6,28.0,0.705333
4,7.0,37.4,1.300000


In [7]:
df["sanitation_gap"] = 1 - df["sanitation_index"]

df[["sanitation_index", "sanitation_gap"]].head()

,sanitation_index,sanitation_gap
0,0.34,0.66
1,0.34,0.66
2,0.34,0.66
3,0.34,0.66
4,0.34,0.66


In [8]:
df["week_number"] = df["date"].dt.isocalendar().week
df["month"] = df["date"].dt.month

df[["date", "week_number", "month"]].head()

,date,week_number,month
0,2024-01-07,1,1
1,2024-01-14,2,1
2,2024-01-21,3,1
3,2024-01-28,4,1
4,2024-02-04,5,2


In [9]:
df_encoded = pd.get_dummies(df, columns=["water_source_type", "disease_type"], drop_first=True)

print(df_encoded.shape)
df_encoded.head()

(1040, 27)


,village_id,village_name,latitude,longitude,population,date,ph_level,turbidity_ntu,residual_chlorine_mg_l,bacterial_contamination,...,environmental_risk_score,sanitation_gap,week_number,month,water_source_type_handpump,water_source_type_pond,water_source_type_river,disease_type_diarrhea,disease_type_hepatitis_a,disease_type_typhoid
0,V001,Village_1,16.57454,74.31185,10470,2024-01-07,6.84,1.95,0.456,0,...,1.088667,0.66,1,1,False,True,False,True,False,False
1,V001,Village_1,16.57454,74.31185,10470,2024-01-14,7.33,2.31,0.462,0,...,0.508000,0.66,2,1,False,True,False,False,False,True
2,V001,Village_1,16.57454,74.31185,10470,2024-01-21,5.94,2.49,0.326,0,...,0.701333,0.66,3,1,False,True,False,True,False,False
3,V001,Village_1,16.57454,74.31185,10470,2024-01-28,6.58,1.78,0.382,0,...,0.705333,0.66,4,1,False,True,False,False,False,False
4,V001,Village_1,16.57454,74.31185,10470,2024-02-04,6.99,2.84,0.274,0,...,1.300000,0.66,5,2,False,True,False,False,True,False


In [10]:
risk_mapping = {"Low": 0, "Medium": 1, "High": 2}
df_encoded["outbreak_risk_level_encoded"] = df_encoded["outbreak_risk_level"].map(risk_mapping)

df_encoded[["outbreak_risk_level", "outbreak_risk_level_encoded"]].head()

,outbreak_risk_level,outbreak_risk_level_encoded
0,High,2
1,High,2
2,Medium,1
3,Low,0
4,Medium,1


In [12]:
df_encoded.to_csv("../data/processed/features_disease_dataset.csv", index=False)
print("Saved. Final shape:", df_encoded.shape)

Saved. Final shape: (1040, 28)


In [18]:
df_encoded.head()

,village_id,village_name,latitude,longitude,population,date,ph_level,turbidity_ntu,residual_chlorine_mg_l,bacterial_contamination,...,sanitation_gap,week_number,month,water_source_type_handpump,water_source_type_pond,water_source_type_river,disease_type_diarrhea,disease_type_hepatitis_a,disease_type_typhoid,outbreak_risk_level_encoded
0,V001,Village_1,16.57454,74.31185,10470,2024-01-07,6.84,1.95,0.456,0,...,0.66,1,1,False,True,False,True,False,False,2
1,V001,Village_1,16.57454,74.31185,10470,2024-01-14,7.33,2.31,0.462,0,...,0.66,2,1,False,True,False,False,False,True,2
2,V001,Village_1,16.57454,74.31185,10470,2024-01-21,5.94,2.49,0.326,0,...,0.66,3,1,False,True,False,True,False,False,1
3,V001,Village_1,16.57454,74.31185,10470,2024-01-28,6.58,1.78,0.382,0,...,0.66,4,1,False,True,False,False,False,False,0
4,V001,Village_1,16.57454,74.31185,10470,2024-02-04,6.99,2.84,0.274,0,...,0.66,5,2,False,True,False,False,True,False,1
